# CDT-III: 2-Stage VCE Training v2 (Improved Regularization)

## Changes from v1
1. **Protein dropout 0.3→0.5**: Stronger regularization on protein-specific components
2. **Weight decay 1e-5→1e-4**: Stronger L2 regularization
3. **DSB threshold 1.0→0.5**: More expressed proteins (~70-80 vs 52) for better generalization

## v1 Results
- RNA r=0.6358 (CDT-II level preserved)
- Protein r=0.4273 (52 expressed proteins, per-cell)
- Protein overfitting observed: Train r=0.60 vs Val r=0.38 in Phase 1

## Architecture (unchanged)
```
DNA [896, 3072] → Projector → Self-Attn(×2)
RNA [2361] → RawEncoder → Self-Attn(×1)
DNA→RNA Cross-Attn

VCE-T(DNA, RNA) → cell_emb_rna [512]        ← CDT-II VCE (fusion transferred!)
    ├── RNA TaskHead → RNA log2FC [2361]      ← L_RNA

Protein [189] → RawEncoder → Self-Attn(×1)
RNA→Protein Cross-Attn

VCE-P(cell_emb_rna, protein) → cell_emb_protein [512]  ← NEW
    └── Protein TaskHead → Protein DSB effect   ← L_Protein (expressed proteins only)
```

### Training Strategy (2 Phases)
- **Phase 1**: CDT-II frozen, train protein path only (300 epochs)
- **Phase 2**: Joint fine-tuning with differential LR (500 epochs)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!ls /content/drive/MyDrive/cdt_data/

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from dataclasses import dataclass
from typing import Dict, Optional, Tuple, List
import numpy as np
import h5py
from pathlib import Path
from scipy.stats import pearsonr
import matplotlib.pyplot as plt
import json
import time

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

In [ ]:
# ============================================================
# Path Configuration
# ============================================================
DRIVE_BASE = Path("/content/drive/MyDrive/cdt_data")
OUTPUT_BASE = Path("/content/drive/MyDrive/cdt_outputs/morris_2stage_vce_v2")
OUTPUT_BASE.mkdir(parents=True, exist_ok=True)

# Input files
CELLLEVEL_WITH_ADT_PATH = DRIVE_BASE / "morris_celllevel_effects_with_adt.h5"
TSS_ENFORMER_PATH = DRIVE_BASE / "morris_28genes_enformer.h5"
SNP_ENFORMER_PATH = DRIVE_BASE / "morris_snp_enformer.h5"

# CDT-II checkpoint (Stage 1.5 best model)
CDT2_CHECKPOINT_PATH = Path("/content/drive/MyDrive/cdt_outputs/morris_crispri_stage1_5") / "cdt_morris_celllevel_best.pt"

# Check files
print("File check:")
for name, path in [
    ("Cell-level + ADT", CELLLEVEL_WITH_ADT_PATH),
    ("TSS Enformer", TSS_ENFORMER_PATH),
    ("CDT-II checkpoint", CDT2_CHECKPOINT_PATH),
]:
    exists = path.exists()
    print(f"  {'✓' if exists else '✗'} {name}: {path}")

## 2. Load Data

In [ ]:
# ============================================================
# Load integrated RNA + ADT cell-level effects
# ============================================================
print("=" * 60)
print("Loading integrated RNA + ADT data...")
print("=" * 60)

with h5py.File(CELLLEVEL_WITH_ADT_PATH, 'r') as f:
    # RNA data
    rna_log2fc = f['log2fc'][:]              # [n_cells, 2361]
    cell_expr = f['cell_expr'][:]            # [n_cells, 2361]
    target_gene_idx = f['target_gene_idx'][:]
    cdt_genes = [g.decode() if isinstance(g, bytes) else g
                 for g in f['gene_names_cdt'][:]]
    target_gene_names = [g.decode() if isinstance(g, bytes) else g
                         for g in f['target_gene_names'][:]]
    ntc_mean_expr = f['ntc_mean_expr'][:]
    train_genes = [g.decode() if isinstance(g, bytes) else g
                   for g in f['train_genes'][:]]
    val_genes = [g.decode() if isinstance(g, bytes) else g
                 for g in f['val_genes'][:]]
    n_cells_per_gene = f['n_cells_per_gene'][:]

    # Protein data
    protein_effect = f['protein_effect'][:]       # [n_cells, 189]
    protein_dsb = f['protein_dsb'][:]            # [n_cells, 189]
    protein_names = [name.decode() if isinstance(name, bytes) else name
                     for name in f['protein_names'][:]]

    if 'protein_matched_mask' in f:
        protein_matched_mask = f['protein_matched_mask'][:]
    else:
        n_cells_attr = f.attrs['n_cells']
        protein_matched_mask = np.ones(n_cells_attr, dtype=bool)
        print("WARNING: 'protein_matched_mask' not found. Assuming all cells have protein data.")

    n_cells = f.attrs['n_cells']
    n_genes = f.attrs['n_genes']
    n_proteins = f.attrs['n_proteins']

print(f"\nRNA:")
print(f"  Cells: {n_cells}")
print(f"  Genes: {n_genes}")
print(f"  log2FC shape: {rna_log2fc.shape}")
print(f"  cell_expr shape: {cell_expr.shape}")
print(f"\nProtein:")
print(f"  Proteins: {n_proteins}")
print(f"  protein_effect shape: {protein_effect.shape}")
print(f"  protein_dsb shape: {protein_dsb.shape}")
print(f"  Cells with protein data: {protein_matched_mask.sum()}/{n_cells}")
print(f"\nTrain genes ({len(train_genes)}): {train_genes[:5]}...")
print(f"Val genes ({len(val_genes)}): {val_genes}")

In [ ]:
# ============================================================
# Load Enformer embeddings
# ============================================================
print("Loading Enformer embeddings...")

# TSS Enformer
with h5py.File(TSS_ENFORMER_PATH, 'r') as f:
    tss_enformer_emb = f['embeddings'][:]  # [28, 896, 3072]
    tss_enformer_genes = [g.decode() if isinstance(g, bytes) else g
                          for g in f['gene_names'][:]]
print(f"  TSS Enformer: {tss_enformer_emb.shape}")

# Build TSS target -> enformer mapping
tss_target_to_enformer = {}
for i, gene in enumerate(target_gene_names):
    if gene in tss_enformer_genes:
        tss_target_to_enformer[i] = tss_enformer_genes.index(gene)
print(f"  TSS matched: {len(tss_target_to_enformer)}/{len(target_gene_names)}")

## 3. ADT Screening: Identify Expressed Proteins

189 ADT proteins include many not expressed in K562. Training on non-expressed proteins (DSB ≈ 0) adds noise and inflates Protein r via trivial 0→0 predictions.

**Strategy**: Screen for proteins with meaningful DSB signal across cells, use only those for loss and evaluation.

In [ ]:
# ============================================================
# ADT Screening: Identify expressed proteins in K562
# ============================================================
# DSB normalization centers isotype controls at 0.
# Proteins with DSB >> 0 are genuinely expressed.
# We use NTC (non-targeting control) cells' mean DSB as the baseline.

# Get NTC cell indices (cells not targeted by any CRISPRi guide)
# In this dataset, we use the mean DSB across all cells as proxy
mean_dsb_per_protein = protein_dsb.mean(axis=0)  # [189]
std_dsb_per_protein = protein_dsb.std(axis=0)    # [189]

# Screening criteria:
# 1. Mean DSB > 1.0 (clearly above background after DSB normalization)
# 2. Std DSB > 0.1 (has some variation — not flatlined)
DSB_MEAN_THRESHOLD = 0.5
DSB_STD_THRESHOLD = 0.1

expressed_mask = (mean_dsb_per_protein > DSB_MEAN_THRESHOLD) & (std_dsb_per_protein > DSB_STD_THRESHOLD)
n_expressed = expressed_mask.sum()

print(f"ADT Screening Results:")
print(f"  Total proteins: {n_proteins}")
print(f"  Mean DSB threshold: {DSB_MEAN_THRESHOLD}")
print(f"  Std DSB threshold: {DSB_STD_THRESHOLD}")
print(f"  Expressed proteins: {n_expressed}")
print(f"  Non-expressed (filtered): {n_proteins - n_expressed}")

# Show expressed proteins
expressed_indices = np.where(expressed_mask)[0]
print(f"\nExpressed proteins ({n_expressed}):")
for idx in expressed_indices:
    print(f"  [{idx:3d}] {protein_names[idx]}: "
          f"mean DSB={mean_dsb_per_protein[idx]:.2f}, "
          f"std={std_dsb_per_protein[idx]:.2f}")

# Show distribution
print(f"\nDSB distribution summary:")
print(f"  All proteins: mean={mean_dsb_per_protein.mean():.3f}, "
      f"range=[{mean_dsb_per_protein.min():.2f}, {mean_dsb_per_protein.max():.2f}]")
print(f"  Expressed: mean={mean_dsb_per_protein[expressed_mask].mean():.3f}")
print(f"  Non-expressed: mean={mean_dsb_per_protein[~expressed_mask].mean():.3f}")

In [ ]:
# ============================================================
# Visualize ADT distribution for screening validation
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Mean DSB distribution
ax = axes[0]
ax.hist(mean_dsb_per_protein, bins=30, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(DSB_MEAN_THRESHOLD, color='red', linestyle='--', label=f'Threshold={DSB_MEAN_THRESHOLD}')
ax.set_xlabel('Mean DSB per protein')
ax.set_ylabel('Count')
ax.set_title(f'ADT Mean DSB Distribution\n({n_expressed} expressed / {n_proteins} total)')
ax.legend()

# 2. Mean vs Std
ax = axes[1]
ax.scatter(mean_dsb_per_protein[~expressed_mask], std_dsb_per_protein[~expressed_mask],
           c='gray', alpha=0.5, s=20, label=f'Non-expressed ({(~expressed_mask).sum()})')
ax.scatter(mean_dsb_per_protein[expressed_mask], std_dsb_per_protein[expressed_mask],
           c='red', alpha=0.8, s=30, label=f'Expressed ({n_expressed})')
ax.axvline(DSB_MEAN_THRESHOLD, color='red', linestyle='--', alpha=0.5)
ax.axhline(DSB_STD_THRESHOLD, color='blue', linestyle='--', alpha=0.5)
ax.set_xlabel('Mean DSB')
ax.set_ylabel('Std DSB')
ax.set_title('ADT Screening: Mean vs Std')
ax.legend()

# 3. Sorted mean DSB with threshold
ax = axes[2]
sorted_means = np.sort(mean_dsb_per_protein)[::-1]
colors = ['red' if v > DSB_MEAN_THRESHOLD else 'gray' for v in sorted_means]
ax.bar(range(n_proteins), sorted_means, color=colors, width=1.0)
ax.axhline(DSB_MEAN_THRESHOLD, color='black', linestyle='--', alpha=0.5)
ax.set_xlabel('Protein rank')
ax.set_ylabel('Mean DSB')
ax.set_title('Proteins ranked by expression')

plt.tight_layout()
plt.savefig(OUTPUT_BASE / 'adt_screening.png', dpi=150, bbox_inches='tight')
plt.show()
print(f"Saved ADT screening plot")

In [ ]:
# ============================================================
# Convert expressed_mask to torch tensor for loss computation
# ============================================================
# This boolean mask selects which of the 189 proteins to use for loss/eval
protein_expressed_mask = torch.from_numpy(expressed_mask)  # [189] bool

print(f"Protein expressed mask: {protein_expressed_mask.shape}")
print(f"  True (expressed): {protein_expressed_mask.sum().item()}")
print(f"  False (filtered): {(~protein_expressed_mask).sum().item()}")
print(f"\nNote: All 189 proteins flow through encoder/self-attn/cross-attn")
print(f"      Only expressed proteins contribute to loss and Pearson r")

## 4. Model Definition

### Configuration and shared components (same as CDT-II / 1-stage VCE)

In [ ]:
# ============================================================
# CDT-III 2-Stage VCE Configuration
# ============================================================

@dataclass
class CDT2StageVCEConfig:
    """CDT-III 2-Stage VCE Model Configuration"""
    # Input dimensions
    dna_dim: int = 3072
    dna_seq_len: int = 896
    n_genes: int = 2361      # RNA genes (2360 + GFI1B)
    n_proteins: int = 189    # ADT proteins (193 - 4 isotype controls)

    # Model dimensions
    hidden_dim: int = 512    # Same as CDT-II
    nhead: int = 8
    dropout: float = 0.3        # CDT-II components (preserved)
    protein_dropout: float = 0.5  # v2: stronger for protein path

    # Self-Attention layers
    dna_self_attn_layers: int = 2
    rna_self_attn_layers: int = 1
    protein_self_attn_layers: int = 1


print("CDT2StageVCEConfig defined")
config = CDT2StageVCEConfig()
print(f"  hidden_dim: {config.hidden_dim}")
print(f"  n_genes: {config.n_genes}")
print(f"  n_proteins: {config.n_proteins}")

In [ ]:
# ============================================================
# Model Components (shared with CDT-II)
# ============================================================

class RawExpressionEncoder(nn.Module):
    """Raw expression -> hidden_dim embeddings.
    Used for both RNA (2361 genes) and Protein (189 proteins).
    """
    def __init__(self, n_features: int, hidden_dim: int, dropout: float = 0.1):
        super().__init__()
        self.n_features = n_features
        self.hidden_dim = hidden_dim

        self.feature_embedding = nn.Embedding(n_features, hidden_dim)
        self.expr_projector = nn.Sequential(
            nn.Linear(1, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Dropout(dropout)
        )
        self.combine = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.LayerNorm(hidden_dim),
            nn.Dropout(dropout)
        )

    def forward(self, expression: torch.Tensor) -> torch.Tensor:
        batch_size = expression.size(0)
        device = expression.device

        feat_ids = torch.arange(self.n_features, device=device)
        feat_emb = self.feature_embedding(feat_ids)
        feat_emb = feat_emb.unsqueeze(0).expand(batch_size, -1, -1)

        expr_emb = self.expr_projector(expression.unsqueeze(-1))

        combined = torch.cat([feat_emb, expr_emb], dim=-1)
        return self.combine(combined)


class SequenceProjector(nn.Module):
    def __init__(self, input_dim: int, output_dim: int, dropout: float = 0.1):
        super().__init__()
        self.linear = nn.Linear(input_dim, output_dim)
        self.norm = nn.LayerNorm(output_dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.norm(self.linear(x)))


class FlashSelfAttentionBlock(nn.Module):
    def __init__(self, d_model: int, nhead: int = 8, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor, return_attn: bool = False):
        batch_size, seq_len, _ = x.shape

        Q = self.q_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(x).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)

        if return_attn:
            scale = self.head_dim ** -0.5
            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
        else:
            attn_out = F.scaled_dot_product_attention(
                Q, K, V,
                dropout_p=self.dropout_p if self.training else 0.0
            )
            attn_weights = None

        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        attn_out = self.out_proj(attn_out)

        x = self.norm1(x + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))

        if return_attn:
            return x, attn_weights
        return x


class FlashCrossAttentionBlock(nn.Module):
    def __init__(self, d_model: int, nhead: int = 8, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = nhead
        self.head_dim = d_model // nhead
        self.dropout_p = dropout

        self.q_proj = nn.Linear(d_model, d_model)
        self.k_proj = nn.Linear(d_model, d_model)
        self.v_proj = nn.Linear(d_model, d_model)
        self.out_proj = nn.Linear(d_model, d_model)

        self.ffn = nn.Sequential(
            nn.Linear(d_model, d_model * 4),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 4, d_model),
            nn.Dropout(dropout)
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, query: torch.Tensor, key_value: torch.Tensor,
                return_attn: bool = False):
        batch_size, query_len, _ = query.shape
        key_len = key_value.shape[1]

        Q = self.q_proj(query).view(batch_size, query_len, self.nhead, self.head_dim).transpose(1, 2)
        K = self.k_proj(key_value).view(batch_size, key_len, self.nhead, self.head_dim).transpose(1, 2)
        V = self.v_proj(key_value).view(batch_size, key_len, self.nhead, self.head_dim).transpose(1, 2)

        if return_attn:
            scale = self.head_dim ** -0.5
            attn_weights = torch.matmul(Q, K.transpose(-2, -1)) * scale
            attn_weights = F.softmax(attn_weights, dim=-1)
            attn_out = torch.matmul(attn_weights, V)
        else:
            attn_out = F.scaled_dot_product_attention(
                Q, K, V,
                dropout_p=self.dropout_p if self.training else 0.0
            )
            attn_weights = None

        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, query_len, self.d_model)
        attn_out = self.out_proj(attn_out)

        x = self.norm1(query + self.dropout(attn_out))
        x = self.norm2(x + self.ffn(x))

        if return_attn:
            return x, attn_weights
        return x


print("Model components defined.")

In [ ]:
# ============================================================
# VCE-T: DNA + RNA Virtual Cell Embedder (exact copy from CDT-II)
# ============================================================
# This is IDENTICAL to CDT-II's VirtualCellEmbedderDNARNA.
# fusion: d_model*2 → d_model*2 → d_model (1024→1024→512)
# ALL weights transfer from CDT-II checkpoint (including fusion!)

class VirtualCellEmbedderDNARNA(nn.Module):
    """VCE-T: DNA + RNA only (2 modalities) — CDT-II architecture"""

    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead

        self.dna_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.rna_query = nn.Parameter(torch.randn(1, 1, d_model))

        self.dna_q_proj = nn.Linear(d_model, d_model)
        self.dna_k_proj = nn.Linear(d_model, d_model)
        self.dna_v_proj = nn.Linear(d_model, d_model)
        self.dna_out_proj = nn.Linear(d_model, d_model)

        self.rna_q_proj = nn.Linear(d_model, d_model)
        self.rna_k_proj = nn.Linear(d_model, d_model)
        self.rna_v_proj = nn.Linear(d_model, d_model)
        self.rna_out_proj = nn.Linear(d_model, d_model)

        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model)
        )

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        batch_size = key_value.size(0)
        seq_len = key_value.size(1)
        query = query.expand(batch_size, -1, -1)

        Q = q_proj(query).view(batch_size, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)

        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, dna_encoded, rna_encoded):
        dna_pooled = self._attention_pool(
            self.dna_query, dna_encoded,
            self.dna_q_proj, self.dna_k_proj, self.dna_v_proj, self.dna_out_proj
        )
        rna_pooled = self._attention_pool(
            self.rna_query, rna_encoded,
            self.rna_q_proj, self.rna_k_proj, self.rna_v_proj, self.rna_out_proj
        )

        concat = torch.cat([dna_pooled, rna_pooled], dim=-1)  # [B, d*2]
        return self.fusion(concat)  # [B, d]


print("VCE-T (VirtualCellEmbedderDNARNA) defined — identical to CDT-II")

In [ ]:
# ============================================================
# VCE-P: Protein Virtual Cell Embedder (NEW in 2-Stage VCE)
# ============================================================
# Takes cell_emb_rna [B, 512] from VCE-T + protein_encoded [B, 189, 512]
# Produces cell_emb_protein [B, 512]
#
# Biological meaning:
#   VCE-T captures nuclear transcription state (DNA→RNA)
#   VCE-P captures cytoplasmic translation state (RNA→Protein)
#   cell_emb_rna flows into VCE-P as the "transcriptome context"

class VirtualCellEmbedderProtein(nn.Module):
    """VCE-P: Protein modality embedder (Stage 2 of 2-Stage VCE)

    Input:
        cell_emb_rna: [B, d_model] from VCE-T
        protein_encoded: [B, n_proteins, d_model] after self-attn + cross-attn

    Output:
        cell_emb_protein: [B, d_model]
    """

    def __init__(self, d_model: int, dropout: float = 0.1):
        super().__init__()
        self.d_model = d_model
        self.nhead = 4
        self.head_dim = d_model // self.nhead

        # Protein attention pooling
        self.protein_query = nn.Parameter(torch.randn(1, 1, d_model))
        self.protein_q_proj = nn.Linear(d_model, d_model)
        self.protein_k_proj = nn.Linear(d_model, d_model)
        self.protein_v_proj = nn.Linear(d_model, d_model)
        self.protein_out_proj = nn.Linear(d_model, d_model)

        # Fusion: cell_emb_rna [512] + protein_pooled [512] → [512]
        # Same structure as VCE-T fusion (d*2 → d*2 → d)
        self.fusion = nn.Sequential(
            nn.Linear(d_model * 2, d_model * 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model * 2, d_model),
            nn.LayerNorm(d_model)
        )

    def _attention_pool(self, query, key_value, q_proj, k_proj, v_proj, out_proj):
        batch_size = key_value.size(0)
        seq_len = key_value.size(1)
        query = query.expand(batch_size, -1, -1)

        Q = q_proj(query).view(batch_size, 1, self.nhead, self.head_dim).transpose(1, 2)
        K = k_proj(key_value).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)
        V = v_proj(key_value).view(batch_size, seq_len, self.nhead, self.head_dim).transpose(1, 2)

        attn_out = F.scaled_dot_product_attention(Q, K, V)
        attn_out = attn_out.transpose(1, 2).contiguous().view(batch_size, 1, self.d_model)
        return out_proj(attn_out).squeeze(1)

    def forward(self, cell_emb_rna, protein_encoded):
        # Attention-pool protein sequence → single vector
        protein_pooled = self._attention_pool(
            self.protein_query, protein_encoded,
            self.protein_q_proj, self.protein_k_proj,
            self.protein_v_proj, self.protein_out_proj
        )  # [B, d_model]

        # Fuse transcriptome context + proteome summary
        concat = torch.cat([cell_emb_rna, protein_pooled], dim=-1)  # [B, d*2]
        return self.fusion(concat)  # [B, d]


print("VCE-P (VirtualCellEmbedderProtein) defined — NEW for 2-Stage VCE")

In [ ]:
# ============================================================
# CDT-III 2-Stage VCE Model
# ============================================================

class CDTTrimodal2StageModel(nn.Module):
    """
    CDT-III: 2-Stage VCE Trimodal Central Dogma Transformer

    Key difference from 1-stage VCE:
    - VCE-T (DNA+RNA) is IDENTICAL to CDT-II → fusion weights transfer 100%
    - VCE-P (RNA→Protein) is NEW → stacked on top of VCE-T output
    - RNA prediction uses cell_emb_rna (from VCE-T)
    - Protein prediction uses cell_emb_protein (from VCE-P)

    Architecture:
        DNA [896, 3072] → Projector → Self-Attn(×2)
        RNA [2361] → RawEncoder → Self-Attn(×1)
        DNA→RNA Cross-Attn

        VCE-T(DNA, RNA) → cell_emb_rna [512]     ← CDT-II (all weights transferred)
            ├── RNA TaskHead → [2361] log2FC

        Protein [189] → RawEncoder → Self-Attn(×1)
        RNA→Protein Cross-Attn

        VCE-P(cell_emb_rna, Protein) → cell_emb_protein [512]  ← NEW
            └── Protein TaskHead → [189] DSB effect
    """

    def __init__(self, config: Optional[CDT2StageVCEConfig] = None):
        super().__init__()
        if config is None:
            config = CDT2StageVCEConfig()
        self.config = config

        # === CDT-II components (ALL transferable) ===
        # DNA
        self.dna_projector = SequenceProjector(
            config.dna_dim, config.hidden_dim, config.dropout)
        self.dna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.dna_self_attn_layers)
        ])
        # RNA
        self.rna_encoder = RawExpressionEncoder(
            config.n_genes, config.hidden_dim, config.dropout)
        self.rna_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.dropout)
            for _ in range(config.rna_self_attn_layers)
        ])
        # DNA→RNA Cross-Attention
        self.dna_to_rna = FlashCrossAttentionBlock(
            config.hidden_dim, config.nhead, config.dropout)
        # VCE-T (CDT-II VCE — fusion d*2→d, fully transferable)
        self.vce_t = VirtualCellEmbedderDNARNA(
            config.hidden_dim, config.dropout)
        # RNA Task Head
        self.rna_task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.dropout),
            nn.Linear(config.hidden_dim, config.n_genes)
        )

        # === CDT-III NEW components ===
        # Protein Encoder
        self.protein_encoder = RawExpressionEncoder(
            config.n_proteins, config.hidden_dim, config.protein_dropout)
        # Protein Self-Attention
        self.protein_self_attn_layers = nn.ModuleList([
            FlashSelfAttentionBlock(config.hidden_dim, config.nhead, config.protein_dropout)
            for _ in range(config.protein_self_attn_layers)
        ])
        # RNA→Protein Cross-Attention
        self.rna_to_protein = FlashCrossAttentionBlock(
            config.hidden_dim, config.nhead, config.protein_dropout)
        # VCE-P (NEW — takes cell_emb_rna + protein)
        self.vce_p = VirtualCellEmbedderProtein(
            config.hidden_dim, config.protein_dropout)
        # Protein Task Head
        self.protein_task_layer = nn.Sequential(
            nn.Linear(config.hidden_dim, config.hidden_dim),
            nn.GELU(),
            nn.Dropout(config.protein_dropout),
            nn.Linear(config.hidden_dim, config.n_proteins)
        )

    def forward(self, dna_emb, rna_expr, protein_expr,
                return_attention: bool = False):
        """
        Args:
            dna_emb: [batch, 896, 3072]
            rna_expr: [batch, n_genes]
            protein_expr: [batch, n_proteins]

        Returns:
            rna_pred: [batch, n_genes]
            protein_pred: [batch, n_proteins]
        """
        attn_maps = {} if return_attention else None

        # === Encode ===
        dna = self.dna_projector(dna_emb)            # [B, 896, 512]
        rna = self.rna_encoder(rna_expr)             # [B, 2361, 512]
        protein = self.protein_encoder(protein_expr)  # [B, 189, 512]

        # === DNA Self-Attention ===
        for i, layer in enumerate(self.dna_self_attn_layers):
            if return_attention:
                dna, attn_w = layer(dna, return_attn=True)
                attn_maps[f'dna_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                dna = layer(dna)

        # === RNA Self-Attention ===
        for i, layer in enumerate(self.rna_self_attn_layers):
            if return_attention:
                rna, attn_w = layer(rna, return_attn=True)
                attn_maps[f'rna_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                rna = layer(rna)

        # === Protein Self-Attention ===
        for i, layer in enumerate(self.protein_self_attn_layers):
            if return_attention:
                protein, attn_w = layer(protein, return_attn=True)
                attn_maps[f'protein_self_attn_{i}'] = attn_w.detach().cpu()
            else:
                protein = layer(protein)

        # === Cross-Attention: DNA → RNA ===
        if return_attention:
            rna, attn_w = self.dna_to_rna(query=rna, key_value=dna, return_attn=True)
            attn_maps['dna_to_rna_cross'] = attn_w.detach().cpu()
        else:
            rna = self.dna_to_rna(query=rna, key_value=dna)

        # === Cross-Attention: RNA → Protein ===
        if return_attention:
            protein, attn_w = self.rna_to_protein(
                query=protein, key_value=rna, return_attn=True)
            attn_maps['rna_to_protein_cross'] = attn_w.detach().cpu()
        else:
            protein = self.rna_to_protein(query=protein, key_value=rna)

        # === Stage 1: VCE-T (DNA + RNA) → cell_emb_rna ===
        cell_emb_rna = self.vce_t(dna, rna)  # [B, 512]

        # === RNA Prediction (from VCE-T output) ===
        rna_pred = self.rna_task_layer(cell_emb_rna)  # [B, 2361]

        # === Stage 2: VCE-P (cell_emb_rna + Protein) → cell_emb_protein ===
        cell_emb_protein = self.vce_p(cell_emb_rna, protein)  # [B, 512]

        # === Protein Prediction (from VCE-P output) ===
        protein_pred = self.protein_task_layer(cell_emb_protein)  # [B, 189]

        if return_attention:
            return rna_pred, protein_pred, attn_maps
        return rna_pred, protein_pred

    # --- Freeze/Unfreeze methods for 2-phase training ---

    def get_cdt2_param_names(self):
        """CDT-II component prefixes (all transferable)."""
        return [
            'dna_projector', 'dna_self_attn_layers',
            'rna_encoder', 'rna_self_attn_layers',
            'dna_to_rna', 'vce_t', 'rna_task_layer'
        ]

    def get_cdt3_param_names(self):
        """CDT-III NEW component prefixes."""
        return [
            'protein_encoder', 'protein_self_attn_layers',
            'rna_to_protein', 'vce_p', 'protein_task_layer'
        ]

    def freeze_cdt2(self):
        """Phase 1: Freeze ALL CDT-II components (including VCE-T + rna_task_layer)."""
        cdt2_prefixes = self.get_cdt2_param_names()
        frozen_count = 0
        for name, param in self.named_parameters():
            if any(name.startswith(prefix) for prefix in cdt2_prefixes):
                param.requires_grad = False
                frozen_count += 1
        print(f"Frozen {frozen_count} CDT-II parameters (incl. VCE-T fusion)")
        print(f"Trainable: {self.get_num_params(trainable_only=True):,} params")

    def unfreeze_all(self):
        """Phase 2: Unfreeze all parameters for joint fine-tuning."""
        for param in self.parameters():
            param.requires_grad = True
        print(f"Unfrozen all {sum(1 for p in self.parameters())} parameters")

    def get_num_params(self, trainable_only: bool = False):
        if trainable_only:
            return sum(p.numel() for p in self.parameters() if p.requires_grad)
        return sum(p.numel() for p in self.parameters())


# Test model creation
print("Testing CDTTrimodal2StageModel...")
test_config = CDT2StageVCEConfig(n_genes=100, n_proteins=20)
test_model = CDTTrimodal2StageModel(test_config)

with torch.no_grad():
    dna_test = torch.randn(2, 896, 3072)
    rna_test = torch.randn(2, 100)
    prot_test = torch.randn(2, 20)
    rna_out, prot_out = test_model(dna_test, rna_test, prot_test)

print(f"  RNA output: {rna_out.shape}")
print(f"  Protein output: {prot_out.shape}")
print(f"  Total params: {test_model.get_num_params():,}")

# Test freeze
test_model.freeze_cdt2()
print(f"  After freeze — trainable: {test_model.get_num_params(trainable_only=True):,}")
test_model.unfreeze_all()

del test_model, dna_test, rna_test, prot_test
print("\nModel test passed!")

## 5. Weight Transfer from CDT-II

Key difference from 1-stage VCE: `vce.*` → `vce_t.*` with **fusion included** (d×2→d dimensions match).

In [ ]:
# ============================================================
# Load CDT-II weights and transfer to 2-Stage VCE model
# ============================================================

def load_cdt2_weights_2stage(model: CDTTrimodal2StageModel, checkpoint_path: Path,
                              device: str = 'cpu') -> Dict[str, int]:
    """
    Transfer CDT-II trained weights to CDT-III 2-Stage VCE model.

    Key mapping:
        CDT-II                    → CDT-III 2-Stage
        dna_projector.*           → dna_projector.*           (direct copy)
        dna_self_attn_layers.*    → dna_self_attn_layers.*    (direct copy)
        rna_encoder.gene_embedding.* → rna_encoder.feature_embedding.* (renamed!)
        rna_encoder.(other).*     → rna_encoder.(other).*     (direct copy)
        rna_self_attn_layers.*    → rna_self_attn_layers.*    (direct copy)
        dna_to_rna.*              → dna_to_rna.*              (direct copy)
        task_layer.*              → rna_task_layer.*           (renamed)
        vce.*                     → vce_t.*                   (renamed, ALL including fusion!)

    Unlike 1-stage VCE, VCE fusion IS transferred (d*2→d dimensions match).
    """
    print(f"Loading CDT-II checkpoint from {checkpoint_path}...")
    cdt2_state = torch.load(checkpoint_path, map_location=device, weights_only=False)

    if 'model_state_dict' in cdt2_state:
        cdt2_state = cdt2_state['model_state_dict']

    model_state = model.state_dict()

    # Build mapping
    key_mapping = {}
    for cdt2_key in cdt2_state.keys():
        # Direct copy (DNA, RNA self-attn, cross-attn)
        if cdt2_key.startswith(('dna_projector.', 'dna_self_attn_layers.',
                                'rna_self_attn_layers.', 'dna_to_rna.')):
            if cdt2_key in model_state:
                key_mapping[cdt2_key] = cdt2_key

        # RNA encoder: gene_embedding → feature_embedding rename
        elif cdt2_key.startswith('rna_encoder.'):
            if 'gene_embedding' in cdt2_key:
                new_key = cdt2_key.replace('gene_embedding', 'feature_embedding')
            else:
                new_key = cdt2_key
            if new_key in model_state:
                key_mapping[cdt2_key] = new_key

        # Rename: task_layer → rna_task_layer
        elif cdt2_key.startswith('task_layer.'):
            new_key = cdt2_key.replace('task_layer.', 'rna_task_layer.')
            if new_key in model_state:
                key_mapping[cdt2_key] = new_key

        # Rename: vce.* → vce_t.* (INCLUDING fusion — dimensions match!)
        elif cdt2_key.startswith('vce.'):
            new_key = cdt2_key.replace('vce.', 'vce_t.')
            if new_key in model_state:
                key_mapping[cdt2_key] = new_key

    # Apply mapping
    loaded_keys = []
    skipped_keys = []

    for cdt2_key, cdt3_key in key_mapping.items():
        if cdt2_state[cdt2_key].shape == model_state[cdt3_key].shape:
            model_state[cdt3_key] = cdt2_state[cdt2_key]
            loaded_keys.append(f"{cdt2_key} → {cdt3_key}")
        else:
            skipped_keys.append(
                f"{cdt2_key}: CDT-II {cdt2_state[cdt2_key].shape} "
                f"vs CDT-III {model_state[cdt3_key].shape}")

    model.load_state_dict(model_state)

    # Identify new parameters
    transferred_cdt3_keys = set(key_mapping.values())
    new_keys = [k for k in model_state.keys() if k not in transferred_cdt3_keys]

    # Keys in CDT-II but not mapped
    unmapped_cdt2 = [k for k in cdt2_state.keys() if k not in key_mapping]

    stats = {
        'loaded': len(loaded_keys),
        'skipped': len(skipped_keys),
        'new': len(new_keys),
        'unmapped_cdt2': len(unmapped_cdt2)
    }

    print(f"\nWeight transfer summary:")
    print(f"  CDT-II keys: {len(cdt2_state)}")
    print(f"  Loaded: {stats['loaded']} parameters")
    print(f"  Skipped (shape mismatch): {stats['skipped']}")
    print(f"  New (random init): {stats['new']}")
    print(f"  Unmapped CDT-II keys: {stats['unmapped_cdt2']}")

    if skipped_keys:
        print(f"\n  Skipped details:")
        for s in skipped_keys:
            print(f"    {s}")

    if unmapped_cdt2:
        print(f"\n  Unmapped CDT-II keys:")
        for k in unmapped_cdt2:
            print(f"    {k}")

    # Show key renames
    renamed = [(c2, c3) for c2, c3 in key_mapping.items() if c2 != c3]
    if renamed:
        print(f"\n  Renamed keys ({len(renamed)}):")
        for c2, c3 in renamed:
            print(f"    {c2} → {c3}")

    # Show fusion transfer status
    fusion_transferred = [k for k in loaded_keys if 'fusion' in k]
    print(f"\n  VCE fusion weights transferred: {len(fusion_transferred)}")

    # Show new component prefixes
    new_prefixes = set()
    for k in new_keys:
        prefix = k.split('.')[0]
        new_prefixes.add(prefix)
    print(f"  New component prefixes: {sorted(new_prefixes)}")

    return stats


print("load_cdt2_weights_2stage() defined (with gene_embedding→feature_embedding fix).")

## 6. Dataset and DataLoader

In [ ]:
# ============================================================
# Trimodal Dataset (same as 1-stage VCE)
# ============================================================

class MorrisTrimodalDataset(Dataset):
    """Cell-level Trimodal Dataset for CDT-III.

    Each sample = one perturbed cell with:
    - DNA: Enformer embedding for the target gene's TSS [896, 3072]
    - RNA: Cell's expression [2361]
    - Protein: Cell's DSB-normalized ADT [189]
    - RNA target: Per-cell log2FC [2361]
    - Protein target: Per-cell protein DSB effect [189]
    """

    def __init__(self, indices, rna_log2fc, cell_expr, protein_dsb_values,
                 protein_effect_values, target_gene_idx, enformer_emb,
                 target_to_enformer, protein_matched_mask):
        self.rna_log2fc = rna_log2fc
        self.cell_expr = cell_expr
        self.protein_dsb = protein_dsb_values
        self.protein_effect = protein_effect_values
        self.target_gene_idx = target_gene_idx
        self.enformer_emb = enformer_emb
        self.target_to_enformer = target_to_enformer

        # Filter: need both valid Enformer AND protein data
        self.valid_indices = [
            idx for idx in indices
            if (target_gene_idx[idx] in target_to_enformer
                and protein_matched_mask[idx])
        ]

        n_filtered = len(indices) - len(self.valid_indices)
        if n_filtered > 0:
            print(f"  Filtered {n_filtered} cells "
                  f"(missing Enformer or protein data)")

    def __len__(self):
        return len(self.valid_indices)

    def __getitem__(self, idx):
        cell_idx = self.valid_indices[idx]
        gene_idx = self.target_gene_idx[cell_idx]
        enf_idx = self.target_to_enformer[gene_idx]

        dna = torch.from_numpy(
            self.enformer_emb[enf_idx].astype(np.float32))
        rna = torch.from_numpy(
            self.cell_expr[cell_idx].astype(np.float32))
        protein = torch.from_numpy(
            self.protein_dsb[cell_idx].astype(np.float32))

        rna_target = torch.from_numpy(
            self.rna_log2fc[cell_idx].astype(np.float32))
        protein_target = torch.from_numpy(
            self.protein_effect[cell_idx].astype(np.float32))

        return dna, rna, protein, rna_target, protein_target


print("MorrisTrimodalDataset defined.")

In [ ]:
# ============================================================
# Create Train/Val Split
# ============================================================

gene_name_to_idx = {name: i for i, name in enumerate(target_gene_names)}

val_gene_indices = set()
for vg in val_genes:
    if vg in gene_name_to_idx:
        val_gene_indices.add(gene_name_to_idx[vg])

train_indices = []
val_indices = []
for i in range(n_cells):
    if target_gene_idx[i] in val_gene_indices:
        val_indices.append(i)
    else:
        train_indices.append(i)

print(f"Train cells: {len(train_indices)}")
print(f"Val cells: {len(val_indices)}")
print(f"Val genes: {val_genes}")

# Create datasets
train_dataset = MorrisTrimodalDataset(
    train_indices, rna_log2fc, cell_expr,
    protein_dsb, protein_effect,
    target_gene_idx, tss_enformer_emb,
    tss_target_to_enformer, protein_matched_mask
)

val_dataset = MorrisTrimodalDataset(
    val_indices, rna_log2fc, cell_expr,
    protein_dsb, protein_effect,
    target_gene_idx, tss_enformer_emb,
    tss_target_to_enformer, protein_matched_mask
)

print(f"\nDataset sizes:")
print(f"  Train: {len(train_dataset)}")
print(f"  Val: {len(val_dataset)}")

# DataLoaders
BATCH_SIZE = 64

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    num_workers=2, pin_memory=True
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    num_workers=2, pin_memory=True
)

# Test batch
dna_b, rna_b, prot_b, rna_t, prot_t = next(iter(train_loader))
print(f"\nBatch shapes:")
print(f"  DNA: {dna_b.shape}")
print(f"  RNA input: {rna_b.shape}")
print(f"  Protein input: {prot_b.shape}")
print(f"  RNA target: {rna_t.shape}")
print(f"  Protein target: {prot_t.shape}")

## 7. Training Configuration and Functions

**Phase 1**: CDT-II frozen → train protein path only. RNA r≈0.64 should appear immediately.
**Phase 2**: All unfrozen → joint fine-tuning with differential LR and RNA guard.

In [ ]:
# ============================================================
# Training Configuration — 2-Phase
# ============================================================

training_config = {
    # Model
    'hidden_dim': 512,
    'nhead': 8,
    'dropout': 0.3,
    'protein_dropout': 0.5,  # v2: stronger regularization for protein path
    'n_genes': n_genes,
    'n_proteins': n_proteins,

    # Data
    'batch_size': BATCH_SIZE,
    'n_train_cells': len(train_dataset),
    'n_val_cells': len(val_dataset),
    'val_genes': val_genes,

    # Phase 1: CDT-II frozen, Protein warmup
    'phase1_epochs': 300,
    'phase1_lr': 1e-3,
    'phase1_patience': 30,
    'phase1_lambda': 1.0,  # CDT-II frozen → only L_Protein matters

    # Phase 2: Joint fine-tuning, differential LR
    'phase2_epochs': 500,
    'phase2_lr_cdt2': 1e-5,
    'phase2_lr_cdt3': 5e-5,
    'phase2_patience': 50,
    'phase2_lambda': 0.1,  # Adjusted after Phase 1
    'phase2_rna_guard': 0.50,  # If RNA r < this, reduce CDT-II LR

    # Common
    'weight_decay': 1e-4,  # v2: 10x stronger L2 regularization

    # ADT screening
    'dsb_mean_threshold': DSB_MEAN_THRESHOLD,
    'dsb_std_threshold': DSB_STD_THRESHOLD,
    'n_expressed_proteins': int(n_expressed),
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
print(f"\n2-Phase Training Config:")
print(f"  Phase 1: CDT-II frozen, protein warmup "
      f"(λ={training_config['phase1_lambda']}, "
      f"{training_config['phase1_epochs']} epochs)")
print(f"  Phase 2: Joint fine-tuning "
      f"(λ={training_config['phase2_lambda']}, "
      f"{training_config['phase2_epochs']} epochs)")
print(f"  Expressed proteins for loss: {training_config['n_expressed_proteins']}/{n_proteins}")

In [ ]:
# ============================================================
# Training Functions (with ADT screening mask)
# ============================================================

rna_criterion = nn.MSELoss()
protein_criterion = nn.MSELoss()


def compute_loss(rna_pred, protein_pred, rna_target, protein_target,
                 lam, prot_mask):
    """Combined loss with expressed-protein masking.

    Args:
        prot_mask: [n_proteins] bool tensor on same device as protein_pred
    """
    l_rna = rna_criterion(rna_pred, rna_target)
    # Protein loss on expressed proteins only
    l_protein = protein_criterion(
        protein_pred[:, prot_mask], protein_target[:, prot_mask])
    return l_rna + lam * l_protein, l_rna, l_protein


def train_epoch(model, loader, optimizer, device, lam, prot_mask):
    model.train()
    total_loss = 0
    total_rna_loss = 0
    total_prot_loss = 0
    all_rna_preds, all_rna_targets = [], []
    all_prot_preds, all_prot_targets = [], []
    n_samples = 0

    for dna, rna, protein, rna_target, protein_target in loader:
        dna = dna.to(device)
        rna = rna.to(device)
        protein = protein.to(device)
        rna_target = rna_target.to(device)
        protein_target = protein_target.to(device)
        bs = dna.size(0)

        optimizer.zero_grad()
        rna_pred, protein_pred = model(dna, rna, protein)

        loss, l_rna, l_prot = compute_loss(
            rna_pred, protein_pred, rna_target, protein_target, lam, prot_mask)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        total_loss += loss.item() * bs
        total_rna_loss += l_rna.item() * bs
        total_prot_loss += l_prot.item() * bs
        n_samples += bs

        all_rna_preds.append(rna_pred.detach().cpu().numpy())
        all_rna_targets.append(rna_target.cpu().numpy())
        # Pearson r on expressed proteins only
        all_prot_preds.append(protein_pred[:, prot_mask.cpu()].detach().cpu().numpy())
        all_prot_targets.append(protein_target[:, prot_mask.cpu()].cpu().numpy())

    all_rna_preds = np.concatenate(all_rna_preds).flatten()
    all_rna_targets = np.concatenate(all_rna_targets).flatten()
    all_prot_preds = np.concatenate(all_prot_preds).flatten()
    all_prot_targets = np.concatenate(all_prot_targets).flatten()

    rna_r, _ = pearsonr(all_rna_preds, all_rna_targets)
    prot_r, _ = pearsonr(all_prot_preds, all_prot_targets)

    return {
        'loss': total_loss / n_samples,
        'rna_loss': total_rna_loss / n_samples,
        'protein_loss': total_prot_loss / n_samples,
        'rna_r': rna_r,
        'protein_r': prot_r,
    }


def evaluate(model, loader, device, lam, prot_mask):
    model.eval()
    total_loss = 0
    total_rna_loss = 0
    total_prot_loss = 0
    all_rna_preds, all_rna_targets = [], []
    all_prot_preds, all_prot_targets = [], []
    n_samples = 0

    with torch.no_grad():
        for dna, rna, protein, rna_target, protein_target in loader:
            dna = dna.to(device)
            rna = rna.to(device)
            protein = protein.to(device)
            rna_target = rna_target.to(device)
            protein_target = protein_target.to(device)
            bs = dna.size(0)

            rna_pred, protein_pred = model(dna, rna, protein)

            loss, l_rna, l_prot = compute_loss(
                rna_pred, protein_pred, rna_target, protein_target, lam, prot_mask)

            total_loss += loss.item() * bs
            total_rna_loss += l_rna.item() * bs
            total_prot_loss += l_prot.item() * bs
            n_samples += bs

            all_rna_preds.append(rna_pred.cpu().numpy())
            all_rna_targets.append(rna_target.cpu().numpy())
            all_prot_preds.append(protein_pred[:, prot_mask.cpu()].cpu().numpy())
            all_prot_targets.append(protein_target[:, prot_mask.cpu()].cpu().numpy())

    all_rna_preds = np.concatenate(all_rna_preds).flatten()
    all_rna_targets = np.concatenate(all_rna_targets).flatten()
    all_prot_preds = np.concatenate(all_prot_preds).flatten()
    all_prot_targets = np.concatenate(all_prot_targets).flatten()

    rna_r, _ = pearsonr(all_rna_preds, all_rna_targets)
    prot_r, _ = pearsonr(all_prot_preds, all_prot_targets)

    return {
        'loss': total_loss / n_samples,
        'rna_loss': total_rna_loss / n_samples,
        'protein_loss': total_prot_loss / n_samples,
        'rna_r': rna_r,
        'protein_r': prot_r,
    }


print("Training functions defined (with expressed-protein masking).")

In [ ]:
# ============================================================
# Create Model and Load CDT-II Weights
# ============================================================

model_config = CDT2StageVCEConfig(
    n_genes=training_config['n_genes'],
    n_proteins=training_config['n_proteins'],
    hidden_dim=training_config['hidden_dim'],
    nhead=training_config['nhead'],
    dropout=training_config['dropout'],
)

model = CDTTrimodal2StageModel(model_config).to(device)
print(f"CDT-III 2-Stage VCE total parameters: {model.get_num_params():,}")

# Move protein mask to device
prot_mask_device = protein_expressed_mask.to(device)

# Load CDT-II weights (including VCE fusion!)
if CDT2_CHECKPOINT_PATH.exists():
    stats = load_cdt2_weights_2stage(model, CDT2_CHECKPOINT_PATH, device=device)
else:
    print("WARNING: CDT-II checkpoint not found. Training from scratch.")
    stats = {'loaded': 0, 'skipped': 0, 'new': 0}

In [ ]:
# ============================================================
# Verify weight transfer: RNA r≈0.64 should appear immediately
# ============================================================
print("Verifying CDT-II weight transfer...")
print("Expected: RNA r ≈ 0.64 (CDT-II performance)")
print()

val_metrics = evaluate(model, val_loader, device, lam=0.0, prot_mask=prot_mask_device)
print(f"Post-transfer validation:")
print(f"  RNA r  = {val_metrics['rna_r']:.4f}  (expect ~0.64)")
print(f"  RNA loss = {val_metrics['rna_loss']:.6f}")
print(f"  Protein r = {val_metrics['protein_r']:.4f}  (random init, expect ~0)")

if val_metrics['rna_r'] > 0.55:
    print(f"\n✓ Weight transfer successful! RNA r={val_metrics['rna_r']:.4f}")
elif val_metrics['rna_r'] > 0.30:
    print(f"\n⚠ Partial transfer. RNA r={val_metrics['rna_r']:.4f} — check VCE fusion")
else:
    print(f"\n✗ Transfer failed! RNA r={val_metrics['rna_r']:.4f} — investigate")

## 8. Phase 1: Protein Warmup (CDT-II Frozen)

CDT-II is frozen. Only protein_encoder, protein_self_attn, rna_to_protein, vce_p, protein_task_layer are trainable.
RNA r≈0.64 is maintained by frozen CDT-II weights. λ=1.0 (only protein loss drives gradient).

In [ ]:
# ============================================================
# Phase 1: Protein Warmup — CDT-II frozen
# ============================================================

CHECKPOINT_PATH = OUTPUT_BASE / "checkpoint_latest.pt"
BEST_MODEL_PATH = OUTPUT_BASE / "cdt_2stage_vce_best.pt"

# History tracking
start_phase = 1
start_epoch = 0
best_loss = float('inf')
best_val_rna_r = -1
best_val_prot_r = -1
patience_counter = 0
history = {
    'phase': [], 'epoch': [],
    'train_loss': [], 'train_rna_loss': [], 'train_protein_loss': [],
    'train_rna_r': [], 'train_protein_r': [],
    'val_loss': [], 'val_rna_loss': [], 'val_protein_loss': [],
    'val_rna_r': [], 'val_protein_r': [],
    'lambda': [],
}

# Check for checkpoint resume
if CHECKPOINT_PATH.exists():
    print("Resuming from checkpoint...")
    ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    best_loss = ckpt.get('best_loss', float('inf'))
    best_val_rna_r = ckpt.get('best_val_rna_r', -1)
    best_val_prot_r = ckpt.get('best_val_prot_r', -1)
    patience_counter = ckpt.get('patience_counter', 0)
    history = ckpt.get('history', history)
    if 'lambda' not in history:
        history['lambda'] = [0.0] * len(history['phase'])
    start_phase = ckpt.get('phase', 1)
    start_epoch = ckpt.get('epoch', 0) + 1
    print(f"  Resumed: Phase {start_phase}, Epoch {start_epoch}")
    print(f"  Best val RNA r={best_val_rna_r:.4f}, Protein r={best_val_prot_r:.4f}")
else:
    print("Starting fresh 2-phase training...")

# ---- Phase 1 ----
if start_phase <= 1:
    lam = training_config['phase1_lambda']
    print("\n" + "=" * 70)
    print(f"PHASE 1: Protein Warmup (CDT-II frozen, λ={lam})")
    print(f"  Loss = L_RNA + {lam} * L_Protein (expressed only)")
    print(f"  CDT-II frozen → RNA r should stay ≈0.64")
    print("=" * 70)

    # Freeze CDT-II (VCE-T + rna_task_layer + all encoders)
    model.freeze_cdt2()

    optimizer_p1 = torch.optim.AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=training_config['phase1_lr'],
        weight_decay=training_config['weight_decay']
    )
    scheduler_p1 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_p1, mode='min', factor=0.5, patience=10
    )

    # Resume optimizer if available
    if CHECKPOINT_PATH.exists() and start_phase == 1:
        ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
        if 'optimizer_state_dict' in ckpt:
            try:
                optimizer_p1.load_state_dict(ckpt['optimizer_state_dict'])
            except:
                print("  Could not resume optimizer, starting fresh")
        if 'scheduler_state_dict' in ckpt:
            try:
                scheduler_p1.load_state_dict(ckpt['scheduler_state_dict'])
            except:
                pass

    phase1_start = start_epoch if start_phase == 1 else 0
    phase1_best_prot_loss = best_loss if start_phase == 1 else float('inf')

    for epoch in range(phase1_start, training_config['phase1_epochs']):
        train_metrics = train_epoch(
            model, train_loader, optimizer_p1, device, lam, prot_mask_device)
        val_metrics = evaluate(
            model, val_loader, device, lam, prot_mask_device)

        # Record
        history['phase'].append(1)
        history['epoch'].append(epoch)
        history['lambda'].append(lam)
        for k in ['loss', 'rna_loss', 'protein_loss', 'rna_r', 'protein_r']:
            history[f'train_{k}'].append(train_metrics[k])
            history[f'val_{k}'].append(val_metrics[k])

        scheduler_p1.step(val_metrics['protein_loss'])
        current_lr = optimizer_p1.param_groups[0]['lr']

        # Print every 5 epochs or on improvement
        if (epoch + 1) % 5 == 0 or val_metrics['protein_loss'] < phase1_best_prot_loss:
            print(f"P1 E{epoch+1}/{training_config['phase1_epochs']}: "
                  f"Train[RNA r={train_metrics['rna_r']:.4f} Prot r={train_metrics['protein_r']:.4f}] "
                  f"Val[RNA r={val_metrics['rna_r']:.4f} Prot r={val_metrics['protein_r']:.4f} "
                  f"prot_loss={val_metrics['protein_loss']:.6f}] "
                  f"lr={current_lr:.1e}")

        # Best model (Phase 1: track protein loss for early stopping)
        if val_metrics['protein_loss'] < phase1_best_prot_loss:
            phase1_best_prot_loss = val_metrics['protein_loss']
            best_loss = val_metrics['loss']
            best_val_rna_r = val_metrics['rna_r']
            best_val_prot_r = val_metrics['protein_r']
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'config': training_config,
                'model_config': {
                    'n_genes': model_config.n_genes,
                    'n_proteins': model_config.n_proteins,
                    'hidden_dim': model_config.hidden_dim,
                    'nhead': model_config.nhead,
                    'dropout': model_config.dropout,
                },
                'val_rna_r': best_val_rna_r,
                'val_protein_r': best_val_prot_r,
                'epoch': epoch,
                'phase': 1,
                'expressed_mask': expressed_mask,
            }, BEST_MODEL_PATH)
            print(f"  → Best P1! Protein r={best_val_prot_r:.4f}, RNA r={best_val_rna_r:.4f}")
        else:
            patience_counter += 1

        # Checkpoint
        torch.save({
            'phase': 1, 'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer_p1.state_dict(),
            'scheduler_state_dict': scheduler_p1.state_dict(),
            'best_loss': best_loss,
            'best_val_rna_r': best_val_rna_r,
            'best_val_prot_r': best_val_prot_r,
            'patience_counter': patience_counter,
            'history': history,
            'config': training_config,
        }, CHECKPOINT_PATH)

        if patience_counter >= training_config['phase1_patience']:
            print(f"\nPhase 1 early stopping at epoch {epoch+1}")
            break

    print(f"\n{'='*70}")
    print(f"Phase 1 complete: RNA r={best_val_rna_r:.4f}, Protein r={best_val_prot_r:.4f}")
    print(f"{'='*70}")

    # Load best Phase 1 model
    ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    print("Loaded best Phase 1 model for Phase 2")

    start_phase = 2
    start_epoch = 0
    patience_counter = 0
    best_loss = float('inf')

## 9. Phase 2: Joint Fine-tuning (All Unfrozen)

Differential LR: CDT-II components get 1e-5, CDT-III components get 5e-5.
RNA guard: if val RNA r drops below 0.50, reduce CDT-II LR further.

In [ ]:
# ============================================================
# Phase 2: Joint Fine-tuning — All unfrozen, differential LR
# ============================================================

if start_phase <= 2:
    lam = training_config['phase2_lambda']
    print("\n" + "=" * 70)
    print(f"PHASE 2: Joint Fine-tuning (All unfrozen, λ={lam})")
    print(f"  CDT-II LR: {training_config['phase2_lr_cdt2']}")
    print(f"  CDT-III LR: {training_config['phase2_lr_cdt3']}")
    print(f"  RNA guard: r < {training_config['phase2_rna_guard']} → reduce CDT-II LR")
    print("=" * 70)

    model.unfreeze_all()

    # Differential LR: CDT-II vs CDT-III
    cdt2_prefixes = model.get_cdt2_param_names()
    cdt2_params = []
    cdt3_params = []
    for name, param in model.named_parameters():
        if any(name.startswith(prefix) for prefix in cdt2_prefixes):
            cdt2_params.append(param)
        else:
            cdt3_params.append(param)

    optimizer_p2 = torch.optim.AdamW([
        {'params': cdt2_params, 'lr': training_config['phase2_lr_cdt2']},
        {'params': cdt3_params, 'lr': training_config['phase2_lr_cdt3']},
    ], weight_decay=training_config['weight_decay'])

    scheduler_p2 = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer_p2, mode='min', factor=0.5, patience=15
    )

    # Resume optimizer if available
    if CHECKPOINT_PATH.exists() and start_phase == 2:
        ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
        if 'optimizer_state_dict' in ckpt:
            try:
                optimizer_p2.load_state_dict(ckpt['optimizer_state_dict'])
            except:
                print("  Could not resume optimizer, starting fresh")

    phase2_start = start_epoch if start_phase == 2 else 0
    phase2_best_loss = best_loss
    rna_guard_triggered = False

    for epoch in range(phase2_start, training_config['phase2_epochs']):
        train_metrics = train_epoch(
            model, train_loader, optimizer_p2, device, lam, prot_mask_device)
        val_metrics = evaluate(
            model, val_loader, device, lam, prot_mask_device)

        # Record
        history['phase'].append(2)
        history['epoch'].append(epoch)
        history['lambda'].append(lam)
        for k in ['loss', 'rna_loss', 'protein_loss', 'rna_r', 'protein_r']:
            history[f'train_{k}'].append(train_metrics[k])
            history[f'val_{k}'].append(val_metrics[k])

        scheduler_p2.step(val_metrics['loss'])
        cdt2_lr = optimizer_p2.param_groups[0]['lr']
        cdt3_lr = optimizer_p2.param_groups[1]['lr']

        # RNA guard: if RNA r drops too low, reduce CDT-II LR
        if (val_metrics['rna_r'] < training_config['phase2_rna_guard']
                and not rna_guard_triggered):
            rna_guard_triggered = True
            new_cdt2_lr = cdt2_lr * 0.1
            optimizer_p2.param_groups[0]['lr'] = new_cdt2_lr
            print(f"  ⚠ RNA GUARD: RNA r={val_metrics['rna_r']:.4f} < "
                  f"{training_config['phase2_rna_guard']} → "
                  f"CDT-II LR: {cdt2_lr:.1e} → {new_cdt2_lr:.1e}")

        # Print
        if (epoch + 1) % 5 == 0 or val_metrics['loss'] < phase2_best_loss:
            print(f"P2 E{epoch+1}/{training_config['phase2_epochs']}: "
                  f"Train[RNA r={train_metrics['rna_r']:.4f} Prot r={train_metrics['protein_r']:.4f}] "
                  f"Val[RNA r={val_metrics['rna_r']:.4f} Prot r={val_metrics['protein_r']:.4f} "
                  f"loss={val_metrics['loss']:.6f}] "
                  f"lr=[{cdt2_lr:.1e}/{cdt3_lr:.1e}]")

        # Best model (Phase 2: combined loss)
        if val_metrics['loss'] < phase2_best_loss:
            phase2_best_loss = val_metrics['loss']
            best_loss = val_metrics['loss']
            best_val_rna_r = val_metrics['rna_r']
            best_val_prot_r = val_metrics['protein_r']
            patience_counter = 0
            torch.save({
                'model_state_dict': model.state_dict(),
                'config': training_config,
                'model_config': {
                    'n_genes': model_config.n_genes,
                    'n_proteins': model_config.n_proteins,
                    'hidden_dim': model_config.hidden_dim,
                    'nhead': model_config.nhead,
                    'dropout': model_config.dropout,
                },
                'val_rna_r': best_val_rna_r,
                'val_protein_r': best_val_prot_r,
                'epoch': epoch,
                'phase': 2,
                'expressed_mask': expressed_mask,
            }, BEST_MODEL_PATH)
            print(f"  → Best P2! RNA r={best_val_rna_r:.4f}, "
                  f"Protein r={best_val_prot_r:.4f}")
        else:
            patience_counter += 1

        # Checkpoint
        torch.save({
            'phase': 2, 'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer_p2.state_dict(),
            'best_loss': best_loss,
            'best_val_rna_r': best_val_rna_r,
            'best_val_prot_r': best_val_prot_r,
            'patience_counter': patience_counter,
            'history': history,
            'config': training_config,
        }, CHECKPOINT_PATH)

        if patience_counter >= training_config['phase2_patience']:
            print(f"\nPhase 2 early stopping at epoch {epoch+1}")
            break

    print(f"\n{'='*70}")
    print(f"Phase 2 complete: RNA r={best_val_rna_r:.4f}, Protein r={best_val_prot_r:.4f}")
    print(f"{'='*70}")

## 10. Training History Visualization

In [ ]:
# ============================================================
# Plot Training History — 2-Phase
# ============================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 10))

epochs = list(range(len(history['phase'])))
phases = np.array(history['phase'])
phase_boundary = np.where(np.diff(phases) != 0)[0]

# Phase colors
colors_by_phase = ['#2196F3' if p == 1 else '#FF5722' for p in phases]

# 1. RNA Loss
ax = axes[0, 0]
ax.plot(epochs, history['train_rna_loss'], 'b-', alpha=0.5, label='Train')
ax.plot(epochs, history['val_rna_loss'], 'r-', label='Val')
for pb in phase_boundary:
    ax.axvline(pb, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('RNA Loss')
ax.set_title('RNA Loss (L_RNA)')
ax.legend()

# 2. Protein Loss
ax = axes[0, 1]
ax.plot(epochs, history['train_protein_loss'], 'b-', alpha=0.5, label='Train')
ax.plot(epochs, history['val_protein_loss'], 'r-', label='Val')
for pb in phase_boundary:
    ax.axvline(pb, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Protein Loss')
ax.set_title('Protein Loss (expressed only)')
ax.legend()

# 3. Combined Loss
ax = axes[0, 2]
ax.plot(epochs, history['train_loss'], 'b-', alpha=0.5, label='Train')
ax.plot(epochs, history['val_loss'], 'r-', label='Val')
for pb in phase_boundary:
    ax.axvline(pb, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Combined Loss')
ax.set_title('Combined Loss (L_RNA + λ * L_Protein)')
ax.legend()

# 4. RNA Pearson r
ax = axes[1, 0]
ax.plot(epochs, history['train_rna_r'], 'b-', alpha=0.5, label='Train')
ax.plot(epochs, history['val_rna_r'], 'r-', label='Val')
ax.axhline(0.64, color='green', linestyle='--', alpha=0.5, label='CDT-II baseline')
ax.axhline(0.50, color='orange', linestyle='--', alpha=0.5, label='RNA guard')
for pb in phase_boundary:
    ax.axvline(pb, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Pearson r')
ax.set_title('RNA Pearson r')
ax.legend()

# 5. Protein Pearson r
ax = axes[1, 1]
ax.plot(epochs, history['train_protein_r'], 'b-', alpha=0.5, label='Train')
ax.plot(epochs, history['val_protein_r'], 'r-', label='Val')
for pb in phase_boundary:
    ax.axvline(pb, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('Pearson r')
ax.set_title('Protein Pearson r (expressed only)')
ax.legend()

# 6. Lambda schedule
ax = axes[1, 2]
ax.plot(epochs, history['lambda'], 'k-', linewidth=2)
for pb in phase_boundary:
    ax.axvline(pb, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Epoch')
ax.set_ylabel('λ')
ax.set_title('Lambda Schedule')

plt.suptitle(f'CDT-III 2-Stage VCE Training\n'
             f'Best: RNA r={best_val_rna_r:.4f}, Protein r={best_val_prot_r:.4f}',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_BASE / 'training_history_2stage.png', dpi=150, bbox_inches='tight')
plt.show()
print("Training history saved.")

## 11. Final Evaluation and Results

In [ ]:
# ============================================================
# Load best model and final evaluation
# ============================================================

print("Loading best model...")
best_ckpt = torch.load(BEST_MODEL_PATH, map_location=device, weights_only=False)
model.load_state_dict(best_ckpt['model_state_dict'])

final_val = evaluate(model, val_loader, device, lam=training_config['phase2_lambda'],
                     prot_mask=prot_mask_device)

print(f"\n{'='*60}")
print(f"FINAL RESULTS — CDT-III 2-Stage VCE")
print(f"{'='*60}")
print(f"  RNA Pearson r:     {final_val['rna_r']:.4f}")
print(f"  Protein Pearson r: {final_val['protein_r']:.4f} (expressed only, n={n_expressed})")
print(f"  RNA Loss:          {final_val['rna_loss']:.6f}")
print(f"  Protein Loss:      {final_val['protein_loss']:.6f}")
print(f"  Combined Loss:     {final_val['loss']:.6f}")
print(f"  Best phase:        {best_ckpt.get('phase', '?')}")
print(f"  Best epoch:        {best_ckpt.get('epoch', '?')}")

# Compare with CDT-II and 1-stage VCE
print(f"\n{'='*60}")
print(f"COMPARISON")
print(f"{'='*60}")
print(f"  CDT-II (baseline):  RNA r=0.64")
print(f"  1-stage VCE (best): RNA r=0.37, Protein r=0.80*")
print(f"  2-stage VCE:        RNA r={final_val['rna_r']:.4f}, "
      f"Protein r={final_val['protein_r']:.4f}")
print(f"  * 1-stage Protein r inflated by non-expressed 0→0 predictions")

In [ ]:
import numpy as np # Ensure numpy is imported for type checking

# ============================================================
# Save results JSON
# ============================================================

results = {
    'model': 'CDT-III 2-Stage VCE v2',
    'architecture': '2-stage VCE (VCE-T + VCE-P)',
    'timestamp': time.strftime('%Y-%m-%d %H:%M:%S'),
    'val_rna_r': float(final_val['rna_r']),
    'val_protein_r': float(final_val['protein_r']),
    'val_rna_loss': float(final_val['rna_loss']),
    'val_protein_loss': float(final_val['protein_loss']),
    'val_combined_loss': float(final_val['loss']),
    'n_expressed_proteins': int(n_expressed),
    'dsb_mean_threshold': DSB_MEAN_THRESHOLD,
    'dsb_std_threshold': DSB_STD_THRESHOLD,
    'expressed_protein_names': [protein_names[i] for i in expressed_indices],
    'best_phase': int(best_ckpt.get('phase', -1)),
    'best_epoch': int(best_ckpt.get('epoch', -1)),
    'training_config': {k: (str(v) if isinstance(v, list) else (int(v) if isinstance(v, np.int64) else v))
                        for k, v in training_config.items()},
    'comparison': {
        'cdt2_rna_r': 0.64,
        '1stage_vce_rna_r': 0.37,
        '1stage_vce_prot_r': 0.80,
        '2stage_vce_rna_r': float(final_val['rna_r']),
        '2stage_vce_prot_r': float(final_val['protein_r']),
    }
}

results_path = OUTPUT_BASE / f"results_2stage_vce_{time.strftime('%Y%m%d_%H%M%S')}.json"
with open(results_path, 'w') as f:
    json.dump(results, f, indent=2)
print(f"Results saved to {results_path}")